# Hướng dẫn chạy Mô hình 3D DATN trên Kaggle qua GitHub
Notebook này tự động thiết lập môi trường ảo Python 3.10, cài đặt các mô hình AI (SDXL, Grounded-SAM2, TRELLIS) và clone mã nguồn của bạn từ GitHub để chạy máy chủ API Ngrok kết nối với Giao diện Web.

## Bước 1: Khởi tạo Python 3.10 và Môi trường ảo (venv)

In [1]:
import subprocess, os

def run(cmd):
    print(f"Executing: {cmd}")
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.stdout: print(r.stdout.strip())
    if r.stderr and r.returncode != 0: print("ERROR:", r.stderr[-500:])
    return r.returncode

VENV = "/opt/venv310"
PY   = f"{VENV}/bin/python"

print("=== 1. Cài đặt Python 3.10 ===")
run("add-apt-repository ppa:deadsnakes/ppa -y")
run("apt-get update -qq")
run("apt-get install -qq python3.10 python3.10-dev python3.10-venv python3.10-distutils")

print("\n=== 2. Khởi tạo môi trường ảo ===")
run(f"python3.10 -m venv {VENV}")
run(f"{PY} --version")

=== 1. Cài đặt Python 3.10 ===
Executing: add-apt-repository ppa:deadsnakes/ppa -y
Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:5 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:7 https://cli.github.com/packages stable/main amd64 Packages [355 B]
Get:8 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:9 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [99.9 kB]
Get:10 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,806 kB]
Get:11 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:12 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubunt

0

In [2]:
!/opt/venv310/bin/pip install -q fastapi uvicorn pyngrok diffusers transformers peft accelerate huggingface_hub spacy trimesh plyfile utils3d pymeshfix pyvista xatlas scipy pillow imageio imageio-ffmpeg opencv-python-headless tqdm easydict rembg[cpu]


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.3/130.3 KB 3.2 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.8/72.8 KB 6.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 15.8 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 41.3 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 680.7/680.7 KB 38.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 KB 38.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 765.1/765.1 KB 48.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.0/31.0 MB 45.7 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 741.0/741.0 KB 59.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 76.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 86.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 261.6/261.6 KB 32.9 MB/s eta 0:00

## Bước 2: Clone Repo GitHub của bạn và cài đặt thư viện

In [3]:
# Nhập link Git chứa code của bạn tại đây
GITHUB_REPO = "https://github.com/Tiens0710/DATN-3d.git"  # <-- Thay thế bằng link Repo thật của bạn

import os, shutil
VENV = "/opt/venv310"
PIP = f"{VENV}/bin/pip"
TEMP_CLONE = "/tmp/my_repo"

# Xoá thư mục cũ nếu có
if os.path.exists(TEMP_CLONE): 
    shutil.rmtree(TEMP_CLONE)

print("=== 1. Tải mã nguồn từ GitHub ===")
!git clone {GITHUB_REPO} {TEMP_CLONE}

# Copy code vào thư mục làm việc chính
if os.path.exists(TEMP_CLONE):
    print("\n=== 2. Đồng bộ mã nguồn vào thư mục làm việc ===")
    !cp -rf {TEMP_CLONE}/* /kaggle/working/
    print("✅ Đã copy files sang /kaggle/working/")
    !ls -l /kaggle/working/
else:
    print("❌ Lỗi: Không thể tải code từ Github. Vui lòng kiểm tra lại link repo.")


=== 1. Tải mã nguồn từ GitHub ===
Cloning into '/tmp/my_repo'...
remote: Enumerating objects: 138, done.
remote: Counting objects: 100% (138/138), done.
remote: Compressing objects: 100% (90/90), done.
remote: Total 138 (delta 91), reused 94 (delta 47), pack-reused 0 (from 0)
Receiving objects: 100% (138/138), 604.90 KiB | 4.73 MiB/s, done.
Resolving deltas: 100% (91/91), done.

=== 2. Đồng bộ mã nguồn vào thư mục làm việc ===
✅ Đã copy files sang /kaggle/working/
total 652
-rw-r--r-- 1 root root  23566 Jul  7 13:50 api-3d.ipynb
-rw-r--r-- 1 root root 546822 Jul  7 13:50 app_background.png
-rw-r--r-- 1 root root  68425 Jul  7 13:50 index.html
-rw-r--r-- 1 root root   3184 Jul  7 13:50 README.md
-rw-r--r-- 1 root root    665 Jul  7 13:50 requirements.txt
-rw-r--r-- 1 root root   3618 Jul  7 13:50 run_pipeline.py
-rw-r--r-- 1 root root   4725 Jul  7 13:50 server.py
drwxr-xr-x 3 root root   4096 Jul  7 13:50 src


## Bước 3: Cài đặt dependencies từ requirements.txt

In [4]:
print("=== 1. Cài đặt các thư viện từ requirements.txt ===")
# Cài đặt PyTorch và CUDA 12.1 trước để đảm bảo tương thích xformers
!{PIP} install -q torch==2.1.0 torchvision==0.16.0 --index-url https://download.pytorch.org/whl/cu121

# Cài đặt các package khác trong requirements.txt
!{PIP} install -q -r /kaggle/working/requirements.txt

# Ép cài đặt các package bổ sung cần thiết khác
!{PIP} install -q xformers==0.0.22.post7 --index-url https://download.pytorch.org/whl/cu121
!{PIP} install -q spconv-cu121==2.3.8 plyfile==0.9 numpy==1.26.4

print("✅ Cài đặt thư viện hoàn tất!")

=== 1. Cài đặt các thư viện từ requirements.txt ===
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 GB 572.2 kB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 85.7 MB/s eta 0:00:00:00:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.2/89.2 MB 16.2 MB/s eta 0:00:0000:0100:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 60.8 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.8/211.8 MB 3.6 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.3/76.3 MB 14.1 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 34.8 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 35.4 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 199.1/199.1 KB 24.9 MB/s eta 0:00:00
     ━━━━━

## Bước 4: Thiết lập và Biên dịch C++ Extension (nvdiffrast & diff-gaussian-rasterization)

In [5]:
import os
env = os.environ.copy()
env["CUDA_HOME"] = "/usr/local/cuda"
env["PATH"] = "/usr/local/cuda/bin:" + env.get("PATH", "")

# Chạy trực tiếp qua ký tự ! để hiển thị thời gian thực toàn bộ quá trình biên dịch
!/opt/venv310/bin/pip install -v --no-cache-dir --no-build-isolation --no-deps --force-reinstall git+https://github.com/NVlabs/nvdiffrast.git


Using pip 22.0.2 from /opt/venv310/lib/python3.10/site-packages/pip (python 3.10)
  Cloning https://github.com/NVlabs/nvdiffrast.git to /tmp/pip-req-build-p6tzt6rp
  Running command git version
  git version 2.34.1
  Running command git clone --filter=blob:none https://github.com/NVlabs/nvdiffrast.git /tmp/pip-req-build-p6tzt6rp
  Cloning into '/tmp/pip-req-build-p6tzt6rp'...
  Running command git rev-parse HEAD
  253ac4fcea7de5f396371124af597e6cc957bfae
  Resolved https://github.com/NVlabs/nvdiffrast.git to commit 253ac4fcea7de5f396371124af597e6cc957bfae
  Running command Preparing metadata (pyproject.toml)
  running dist_info
  creating /tmp/pip-modern-metadata-lfic5z49/nvdiffrast.egg-info
  writing /tmp/pip-modern-metadata-lfic5z49/nvdiffrast.egg-info/PKG-INFO
  writing dependency_links to /tmp/pip-modern-metadata-lfic5z49/nvdiffrast.egg-info/dependency_links.txt
  writing requirements to /tmp/pip-modern-metadata-lfic5z49/nvdiffrast.egg-info/requires.txt
  writing top-level names to

In [6]:
import subprocess, os, shutil

VENV = "/opt/venv310"
PIP  = f"{VENV}/bin/pip"
PY   = f"{VENV}/bin/python"

# 1. Đảm bảo import torch bình thường, nếu không thì tự sửa lỗi numpy
print("=== 1. Kiểm tra môi trường PyTorch ===")
check_torch = subprocess.run([PY, "-c", "import torch; print('PyTorch OK:', torch.__version__)"], capture_output=True, text=True)
if check_torch.returncode != 0:
    print("❌ Lỗi import PyTorch! Đang cài lại numpy...")
    subprocess.run(f"{PIP} install -q --force-reinstall numpy==1.26.4", shell=True)
else:
    print("  ✓ PyTorch hoạt động tốt.")

# 2. Đảm bảo cài đặt wheel và setuptools hỗ trợ bdist_wheel
print("\n=== 2. Kiểm tra bộ cài build-backend (wheel/setuptools) ===")
subprocess.run(f"{PIP} install -q wheel setuptools", shell=True)

# 3. Cài đặt diff-gaussian-rasterization (Mip-Splatting)
print("\n=== 3. Cài đặt diff-gaussian-rasterization ===")
if os.path.exists("/tmp/mip-splatting"):
    shutil.rmtree("/tmp/mip-splatting")
!git clone --recursive https://github.com/autonomousvision/mip-splatting.git /tmp/mip-splatting
!{PIP} install -q --no-build-isolation --no-cache-dir /tmp/mip-splatting/submodules/diff-gaussian-rasterization

# 4. Vá lỗi PyTorch cpp_extension
print("\n=== 4. Vá lỗi PyTorch cpp_extension để bỏ qua check CUDA ===")
cpp_ext_path = f"{VENV}/lib/python3.10/site-packages/torch/utils/cpp_extension.py"
if os.path.exists(cpp_ext_path):
    try:
        with open(cpp_ext_path, "r", encoding="utf-8") as f:
            code = f.read()
        target = "def _check_cuda_version(compiler_name, compiler_version):"
        patched = "def _check_cuda_version(compiler_name, compiler_version):\n    return  # Patched by Antigravity to bypass CUDA version check"
        if target in code and "Patched by Antigravity" not in code:
            code = code.replace(target, patched)
            with open(cpp_ext_path, "w", encoding="utf-8") as f:
                f.write(code)
            print("  ✓ Vá lỗi file cpp_extension.py thành công!")
        else:
            print("  ✓ File cpp_extension.py đã được vá lỗi từ trước.")
    except Exception as e:
        print("  ❌ Lỗi khi vá file cpp_extension.py:", e)

# 5. Biên dịch nvdiffrast
print("\n=== 5. Biên dịch nvdiffrast ===")
env = os.environ.copy()
cuda_path = "/usr/local/cuda"
if os.path.exists(cuda_path):
    env["CUDA_HOME"] = cuda_path
    env["PATH"] = f"{cuda_path}/bin:" + env.get("PATH", "")

print("Đang tiến hành compile nvdiffrast...")
res = subprocess.run(
    f"{PIP} install -v --no-cache-dir --no-build-isolation --no-deps --force-reinstall git+https://github.com/NVlabs/nvdiffrast.git",
    shell=True, env=env, capture_output=True, text=True
)
if res.returncode == 0:
    print("✅ Cài đặt nvdiffrast THÀNH CÔNG!")
else:
    print("❌ Biên dịch lỗi. STDERR:", res.stderr[-500:])


=== 1. Kiểm tra môi trường PyTorch ===
  ✓ PyTorch hoạt động tốt.

=== 2. Kiểm tra bộ cài build-backend (wheel/setuptools) ===

=== 3. Cài đặt diff-gaussian-rasterization ===
Cloning into '/tmp/mip-splatting'...
remote: Enumerating objects: 1678, done.
remote: Counting objects: 100% (82/82), done.
remote: Compressing objects: 100% (43/43), done.
remote: Total 1678 (delta 48), reused 39 (delta 39), pack-reused 1596 (from 1)
Receiving objects: 100% (1678/1678), 21.17 MiB | 41.62 MiB/s, done.
Resolving deltas: 100% (890/890), done.
  Preparing metadata (setup.py) ... done

=== 4. Vá lỗi PyTorch cpp_extension để bỏ qua check CUDA ===
  ✓ File cpp_extension.py đã được vá lỗi từ trước.

=== 5. Biên dịch nvdiffrast ===
Đang tiến hành compile nvdiffrast...
✅ Cài đặt nvdiffrast THÀNH CÔNG!


## Bước 5: Cài đặt Repo TRELLIS & Kiểm tra đăng nhập HuggingFace

In [7]:
# Tải trực tiếp thư mục TRELLIS về thư mục làm việc chính trên Kaggle
!GIT_LFS_SKIP_SMUDGE=1 git clone https://huggingface.co/spaces/trellis-community/TRELLIS /kaggle/working/TRELLIS
print("✅ ĐÃ TẢI XONG THƯ MỤC TRELLIS HỢP LỆ!")


Cloning into '/kaggle/working/TRELLIS'...
remote: Enumerating objects: 402, done.
remote: Counting objects: 100% (11/11), done.
remote: Compressing objects: 100% (10/10), done.
remote: Total 402 (delta 5), reused 1 (delta 1), pack-reused 391 (from 1)
Receiving objects: 100% (402/402), 31.69 MiB | 42.97 MiB/s, done.
Resolving deltas: 100% (122/122), done.
✅ ĐÃ TẢI XONG THƯ MỤC TRELLIS HỢP LỆ!


In [8]:
!/opt/venv310/bin/pip install -q huggingface_hub spacy trimesh

In [9]:
!/opt/venv310/bin/python -m spacy download en_core_web_sm


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 28.9 MB/s eta 0:00:0000:0100:01
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


In [10]:
# !/opt/venv310/bin/python /kaggle/working/server.py


In [11]:
# import time, subprocess
# from pyngrok import ngrok

# # 1. Điền mã ngrok authtoken của bạn tại đây
# NGROK_TOKEN = "3Fj2536cJmPH9jP4D46KqQheptK_2rGfjLmMhfZcZ9UyVKRrB"

# # Thiết lập Token trực tiếp bằng hàm Python (Không dùng lệnh shell)
# ngrok.set_auth_token(NGROK_TOKEN)

# # 2. Khởi động FastAPI server chạy ngầm dưới nền ở cổng 8000
# print("🚀 Đang khởi động FastAPI Server ở cổng 8000...")
# cmd_server = "/opt/venv310/bin/python /kaggle/working/server.py"

# server_proc = subprocess.Popen(cmd_server.split(), stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
# time.sleep(4)

# # 3. Tạo đường truyền công khai (Tunnel) từ Ngrok kết nối với cổng 8000
# try: 
#     ngrok.kill() # Tắt các cổng thừa
#     time.sleep(1)
#     public_url = ngrok.connect(8000)
#     print("\n" + "="*60)
#     print("🎉 KHỞI ĐỘNG HỆ THỐNG THÀNH CÔNG!")
#     print(f"👉 Đường dẫn API Ngrok của bạn: {public_url}")
#     print("="*60 + "\n")
#     print("Hãy copy đường dẫn trên dán vào ô 'Kaggle API URL' ở giao diện index.html trên máy tính của bạn.")
# except Exception as e:
#     print(f"❌ Lỗi khi mở cổng ngrok: {e}")


In [12]:
# 1. Tải và cài đặt GroundingDINO từ mã nguồn (Mất khoảng 1 phút để biên dịch C++)
!git clone https://github.com/IDEA-Research/GroundingDINO.git /tmp/GroundingDINO
!CUDA_HOME=/usr/local/cuda /opt/venv310/bin/pip install -q -e /tmp/GroundingDINO

# 2. Tải và cài đặt Segment Anything 2 (SAM2)
!git clone https://github.com/facebookresearch/segment-anything-2.git /tmp/segment-anything-2
!/opt/venv310/bin/pip install -q -e /tmp/segment-anything-2

# 3. Tạo thư mục và tải Checkpoint của GroundingDINO
!mkdir -p /kaggle/working/groundingdino_ckpt
!wget -q -O /kaggle/working/groundingdino_ckpt/groundingdino_swint_ogc.pth https://github.com/IDEA-Research/GroundingDINO/releases/download/v0.1.0-alpha/groundingdino_swint_ogc.pth
!cp /tmp/GroundingDINO/groundingdino/config/GroundingDINO_SwinT_OGC.py /kaggle/working/groundingdino_ckpt/

# 4. Tạo thư mục và tải Checkpoint của SAM2
!mkdir -p /kaggle/working/sam2_ckpt
!wget -q -O /kaggle/working/sam2_ckpt/sam2_hiera_small.pt https://dl.fbaipublicfiles.com/segment_anything_2/072824/sam2_hiera_small.pt

print("✅ ĐÃ CÀI ĐẶT VÀ TẢI CHECKPOINTS GROUNDED-SAM2 THÀNH CÔNG!")


Cloning into '/tmp/GroundingDINO'...
remote: Enumerating objects: 463, done.
remote: Counting objects: 100% (147/147), done.
remote: Compressing objects: 100% (57/57), done.
remote: Total 463 (delta 108), reused 90 (delta 90), pack-reused 316 (from 1)
Receiving objects: 100% (463/463), 12.88 MiB | 31.62 MiB/s, done.
Resolving deltas: 100% (239/239), done.
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 MB 7.1 MB/s eta 0:00:00:00:010:01m
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.5/472.5 KB 10.8 MB/s eta 0:00:0000:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 280.2/280.2 KB 10.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 12.4 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 256.2/256.2 KB 8.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.7/102.7 KB 10.5 MB/s eta 0:00:00
Cloning into '/tmp/segment-anything-2'...
remote: Enumerating objects: 1107, done.
remote: Co

## Bước 6: Khởi chạy API Server và tạo link kết nối Ngrok

In [13]:
!pip install pyngrok

In [14]:
# 1. Gỡ cài đặt bản lỗi xung đột
!/opt/venv310/bin/pip uninstall -y groundingdino transformers diffusers

# 2. Cài đặt lại đúng phiên bản để chạy được SD3.5 Medium và các mô hình khác ổn định
!/opt/venv310/bin/pip install --force-reinstall groundingdino-py timm==0.9.16 supervision addict yapf
!/opt/venv310/bin/pip install --force-reinstall diffusers==0.31.0 transformers==4.45.2 accelerate==0.34.2 peft==0.13.2

print("✅ ĐÃ ĐỒNG BỘ VÀ KHÔI PHỤC MÔI TRƯỜNG GIỐNG 100% NOTEBOOK GỐC!")


Found existing installation: groundingdino 0.1.0
Uninstalling groundingdino-0.1.0:
  Successfully uninstalled groundingdino-0.1.0
Found existing installation: transformers 4.41.2
Uninstalling transformers-4.41.2:
  Successfully uninstalled transformers-4.41.2
Found existing installation: diffusers 0.30.3
Uninstalling diffusers-0.30.3:
  Successfully uninstalled diffusers-0.30.3
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 82.3/82.3 KB 2.0 MB/s eta 0:00:0000:01
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 34.2 MB/s eta 0:00:00a 0:00:01
  Using cached supervision-0.29.1-py3-none-any.whl (280 kB)
  Using cached addict-2.4.0-py3-none-any.whl (3.8 kB)
  Using cached yapf-0.43.0-py3-none-any.whl (256 kB)
  Using cached safetensors-0.8.0-cp310-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (516 kB)
  Using cached pyyaml-6.0.3-cp310-cp310-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl (770 kB)
  Using cached torch

In [15]:
# === ĐÃ VÔ HIỆU HÓA ĐỂ TRÁNH DOWNGRADE NHẦM PHIÊN BẢN DIFFUSERS (GÂY LỖI SHAPE MISMATCH CHO SD3.5) ===
# Vui lòng sử dụng các cell cài đặt diffusers==0.31.0 ở trên.

# ##  # 1# .#  # G# ỡ#  # c# à# i#  # đ# ặ# t#  # b# ả# n#  # l# ỗ# i#  # x# u# n# g#  # đ# ộ# t# 
# !# /# o# p# t# /# v# e# n# v# 3# 1# 0# /# b# i# n# /# p# i# p#  # u# n# i# n# s# t# a# l# l#  # -# y#  # g# r# o# u# n# d# i# n# g# d# i# n# o#  # t# r# a# n# s# f# o# r# m# e# r# s#  # d# i# f# f# u# s# e# r# s# 
# 
# ##  # 2# .#  # C# à# i#  # đ# ặ# t#  # l# ạ# i#  # đ# ú# n# g#  # p# h# i# ê# n#  # b# ả# n#  # t# ừ#  # N# o# t# e# b# o# o# k#  # g# ố# c#  # c# ủ# a#  # b# ạ# n# 
# !# /# o# p# t# /# v# e# n# v# 3# 1# 0# /# b# i# n# /# p# i# p#  # i# n# s# t# a# l# l#  # -# -# f# o# r# c# e# -# r# e# i# n# s# t# a# l# l#  # g# r# o# u# n# d# i# n# g# d# i# n# o# -# p# y#  # t# i# m# m# =# =# 0# .# 9# .# 1# 6#  # s# u# p# e# r# v# i# s# i# o# n#  # a# d# d# i# c# t#  # y# a# p# f# 
# !# /# o# p# t# /# v# e# n# v# 3# 1# 0# /# b# i# n# /# p# i# p#  # i# n# s# t# a# l# l#  # -# -# f# o# r# c# e# -# r# e# i# n# s# t# a# l# l#  # d# i# f# f# u# s# e# r# s# =# =# 0# .# 3# 0# .# 3#  # t# r# a# n# s# f# o# r# m# e# r# s# =# =# 4# .# 4# 1# .# 2#  # a# c# c# e# l# e# r# a# t# e# =# =# 0# .# 3# 0# .# 1# 
# 
# p# r# i# n# t# (# "# ✅#  # Đ# Ã#  # Đ# Ồ# N# G#  # B# Ộ#  # V# À#  # K# H# Ô# I#  # P# H# Ụ# C#  # M# Ô# I#  # T# R# Ư# Ờ# N# G#  # G# I# Ố# N# G#  # 1# 0# 0# %#  # N# O# T# E# B# O# O# K#  # G# Ố# C# !# "# )# 


Found existing installation: transformers 4.45.2
Uninstalling transformers-4.45.2:
  Successfully uninstalled transformers-4.45.2
Found existing installation: diffusers 0.31.0
Uninstalling diffusers-0.31.0:
  Successfully uninstalled diffusers-0.31.0
  Using cached groundingdino_py-0.4.0-py2.py3-none-any.whl
  Using cached timm-0.9.16-py3-none-any.whl (2.2 MB)
  Using cached supervision-0.29.1-py3-none-any.whl (280 kB)
  Using cached addict-2.4.0-py3-none-any.whl (3.8 kB)
  Using cached yapf-0.43.0-py3-none-any.whl (256 kB)
  Using cached torch-2.12.1-cp310-cp310-manylinux_2_28_x86_64.whl (532.1 MB)
  Using cached torchvision-0.27.1-cp310-cp310-manylinux_2_28_x86_64.whl (7.7 MB)
  Using cached safetensors-0.8.0-cp310-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (516 kB)
  Using cached pyyaml-6.0.3-cp310-cp310-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl (770 kB)
  Using cached huggingface_hub-1.22.0-py3-none-any.whl (765 kB)
  Using cached pycocotools-2.0

In [16]:
# 1. Đảm bảo setuptools đúng phiên bản hỗ trợ compile C++
!/opt/venv310/bin/pip install -q --force-reinstall setuptools==69.5.1 wheel ninja

# 2. Biên dịch nvdiffrast ép buộc dùng PyTorch 2.1.0 và CUDA hiện tại (không dùng cache cũ)
!CUDA_HOME=/usr/local/cuda PATH=/usr/local/cuda/bin:$PATH /opt/venv310/bin/pip install --force-reinstall --no-deps --no-cache-dir --no-build-isolation git+https://github.com/NVlabs/nvdiffrast.git

print("✅ ĐÃ BIÊN DỊCH VÀ KHÔI PHỤC NVDIFFRAST THÀNH CÔNG!")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 894.6/894.6 KB 12.5 MB/s eta 0:00:0000:0100:01
  Cloning https://github.com/NVlabs/nvdiffrast.git to /tmp/pip-req-build-etn_rv74
  Running command git clone --filter=blob:none --quiet https://github.com/NVlabs/nvdiffrast.git /tmp/pip-req-build-etn_rv74
  Resolved https://github.com/NVlabs/nvdiffrast.git to commit 253ac4fcea7de5f396371124af597e6cc957bfae
  Preparing metadata (pyproject.toml) ... done
  error: subprocess-exited-with-error
  
  × Building wheel for nvdiffrast (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for nvdiffrast
Failed to build nvdiffrast
ERROR: Could not build wheels for nvdiffrast, which is required to install pyproject.toml-based projects
✅ ĐÃ BIÊN DỊCH VÀ KHÔI PHỤC NVDIFFRAST THÀNH CÔNG!


In [17]:
# 1. Tải và cài đặt trực tiếp file biên dịch sẵn của nvdiffrast (bỏ qua bước tự build wheel)
!/opt/venv310/bin/pip install --force-reinstall --no-deps https://github.com/camenduru/wheels/releases/download/colab/nvdiffrast-0.3.1-py3-none-any.whl

# 2. Chạy lệnh kiểm tra import thử trong môi trường ảo
!/opt/venv310/bin/python -c "import nvdiffrast.torch as dr; print('🎉 KẾT QUẢ: NVDIFFRAST IMPORT THÀNH CÔNG!')"


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.8/137.8 KB 3.2 MB/s eta 0:00:00a 0:00:01
  Attempting uninstall: nvdiffrast
    Found existing installation: nvdiffrast 0.4.0
    Uninstalling nvdiffrast-0.4.0:
      Successfully uninstalled nvdiffrast-0.4.0
🎉 KẾT QUẢ: NVDIFFRAST IMPORT THÀNH CÔNG!


In [18]:
!/opt/venv310/bin/pip install igraph==0.11.8
# Ép cài đặt lại PyTorch 2.1.0 và xformers đồng bộ từ kho của PyTorch
!/opt/venv310/bin/pip install --force-reinstall torch==2.1.0 torchvision==0.16.0 xformers==0.0.22.post7 --index-url https://download.pytorch.org/whl/cu121



Looking in indexes: https://download.pytorch.org/whl/cu121
  Using cached https://download-r2.pytorch.org/whl/cu121/torch-2.1.0%2Bcu121-cp310-cp310-linux_x86_64.whl (2200.6 MB)
  Using cached https://download-r2.pytorch.org/whl/cu121/torchvision-0.16.0%2Bcu121-cp310-cp310-linux_x86_64.whl (7.0 MB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.8/211.8 MB 6.7 MB/s eta 0:00:0000:0100:01
  Using cached networkx-3.4.2-py3-none-any.whl (1.7 MB)
  Using cached https://download-r2.pytorch.org/whl/triton-2.1.0-0-cp310-cp310-manylinux2014_x86_64.manylinux_2_17_x86_64.whl (89.2 MB)
  Using cached sympy-1.14.0-py3-none-any.whl (6.3 MB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.4/203.4 KB 4.5 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.6/44.6 KB 5.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.9/134.9 KB 19.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.8/62.8 KB 7.6 MB/s eta 0:00:00
  Using cached numpy-2.2.6-cp310-c

In [19]:
!/opt/venv310/bin/pip install --force-reinstall numpy==1.26.4


  Using cached numpy-1.26.4-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (18.2 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 2.2.6
    Uninstalling numpy-2.2.6:
      Successfully uninstalled numpy-2.2.6
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sam-2 1.0 requires torch>=2.5.1, but you have torch 2.1.0+cu121 which is incompatible.
sam-2 1.0 requires torchvision>=0.20.1, but you have torchvision 0.16.0+cu121 which is incompatible.
opencv-python 5.0.0.93 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.


In [20]:
# Cài đặt chính xác phiên bản commit của utils3d giống trong Notebook của bạn
!/opt/venv310/bin/pip install --force-reinstall --no-build-isolation git+https://github.com/EasternJournalist/utils3d.git@9a4eb15e4021b67b12c460c7057d642626897ec8


  Cloning https://github.com/EasternJournalist/utils3d.git (to revision 9a4eb15e4021b67b12c460c7057d642626897ec8) to /tmp/pip-req-build-yotorrx6
  Running command git clone --filter=blob:none --quiet https://github.com/EasternJournalist/utils3d.git /tmp/pip-req-build-yotorrx6
  Running command git rev-parse -q --verify 'sha^9a4eb15e4021b67b12c460c7057d642626897ec8'
  Running command git fetch -q https://github.com/EasternJournalist/utils3d.git 9a4eb15e4021b67b12c460c7057d642626897ec8
  Running command git checkout -q 9a4eb15e4021b67b12c460c7057d642626897ec8
  Resolved https://github.com/EasternJournalist/utils3d.git to commit 9a4eb15e4021b67b12c460c7057d642626897ec8
  Preparing metadata (pyproject.toml) ... done
  Using cached numpy-2.2.6-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (16.8 MB)
  Using cached plyfile-1.1.4-py3-none-any.whl (36 kB)
  Using cached scipy-1.15.3-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (37.7 MB)
  Using cached moderngl-5.12.0-

In [21]:
import time, subprocess, os
from pyngrok import ngrok

# 1. Thay thế mã ngrok authtoken của bạn ở đây
NGROK_TOKEN = "3Fj2536cJmPH9jP4D46KqQheptK_2rGfjLmMhfZcZ9UyVKRrB"  # <-- THAY THẾ BẰNG TOKEN NGROK THẬT CỦA BẠN
!/opt/venv310/bin/ngrok config add-authtoken {NGROK_TOKEN}

# 2. Khởi động FastAPI server chạy ngầm dưới nền ở cổng 8000 và ghi log
print("🚀 Đang khởi động FastAPI Server ở cổng 8000...")
cmd_server = "/opt/venv310/bin/python /kaggle/working/server.py"

# Ghi log ra file để theo dõi quá trình self-healing và khởi động
log_path = "/kaggle/working/server.log"
with open(log_path, 'w') as log_file:
    log_file.write("=== STARTING SERVER ===\n")

log_file = open(log_path, 'a')
server_proc = subprocess.Popen(cmd_server.split(), stdout=log_file, stderr=subprocess.STDOUT)

# Tăng thời gian chờ để server cài đặt setuptools và tự khởi động lại nếu cần
print("⏳ Đang chờ server thiết lập và mở cổng 8000...")
time.sleep(12)

# Đọc và hiển thị nội dung log hiện tại của server
if os.path.exists(log_path):
    with open(log_path, 'r') as f:
        print("\n=== NHẬT KÝ KHỞI ĐỘNG SERVER (SERVER LOG) ===")
        print(f.read())
        print("===============================================\n")

# 3. Tạo đường truyền công khai (Tunnel) từ Ngrok kết nối với cổng 8000
try: 
    ngrok.kill() # Tắt các cổng thừa
    time.sleep(1)
    public_url = ngrok.connect(8000)
    print("\n" + "="*60)
    print("🎉 KHỞI ĐỘNG HỆ THỐNG THÀNH CÔNG!")
    print(f"👉 Đường dẫn API Ngrok của bạn: {public_url}")
    print("="*60 + "\n")
    print("Hãy copy đường dẫn trên dán vào ô 'Kaggle API URL' ở giao diện index.html trên máy tính của bạn.")
except Exception as e:
    print(f"❌ Lỗi khi mở cổng ngrok: {e}")


Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml                                
🚀 Đang khởi động FastAPI Server ở cổng 8000...

🎉 KHỞI ĐỘNG HỆ THỐNG THÀNH CÔNG!
👉 Đường dẫn API Ngrok của bạn: NgrokTunnel: "https://ellipse-cake-endless.ngrok-free.dev" -> "http://localhost:8000"

Hãy copy đường dẫn trên dán vào ô 'Kaggle API URL' ở giao diện index.html trên máy tính của bạn.
